# 2D FDTD — CuPy Operation-Level Profiling

**Authors:** Vraj Patel (241110080), Vardhman Dwivedi (241060033)  
**Course:** IDC 606 — Fast Computational Hydrodynamics, IIT Kanpur

Profiles **every step** from `cupy_operations_breakdown.md`:
1. Kernel definition (lazy RawKernel)
2. GPU memory allocation
3. Pulse compute on CPU + H→D upload
4. Scalar casting + launch config
5. First kernel launch (triggers JIT compilation)
6. Simulation loop — per-step breakdown of H kernel, Ez kernel, energy kernel, sync, D→H

Results saved to `cupy_profile_results.json` for offline analysis.

In [ ]:
!pip install cupy-cuda12x -q 2>/dev/null || pip install cupy-cuda11x -q 2>/dev/null
!nvidia-smi | head -20

In [ ]:
import numpy as np
import cupy as cp
import time
import json

print(f'CuPy {cp.__version__}')
gpu_props = cp.cuda.runtime.getDeviceProperties(0)
gpu_name = gpu_props['name'].decode() if isinstance(gpu_props['name'], bytes) else gpu_props['name']
print(f'GPU: {gpu_name}')

## Setup (not profiled — shared constants)

In [ ]:
c0   = 3.0e8
mu0  = 4.0e-7 * np.pi
eps0 = 1.0 / (mu0 * c0**2)

Nx, Ny = 200, 200
dx = dy = 1e-3
courant = 0.99
dt = courant / (c0 * np.sqrt(1.0/dx**2 + 1.0/dy**2))
n_steps = 400

src_x, src_y = Nx // 2, Ny // 2
spread = 12.0 * dt
t0 = 3.0 * spread

snapshot_interval = 40
snapshot_steps = set(range(0, n_steps, snapshot_interval))
snapshot_steps.add(n_steps - 1)

flops_per_step = Nx*(Ny-1)*3 + (Nx-1)*Ny*3 + (Nx-1)*(Ny-1)*7
flops_total = flops_per_step * n_steps

# Profile storage
profile = {
    'gpu': gpu_name,
    'grid': f'{Nx}x{Ny}',
    'steps': n_steps,
    'flops_per_step': flops_per_step,
    'flops_total': flops_total,
    'setup': {},
    'per_step': [],
}

print(f'Grid: {Nx}x{Ny}, Steps: {n_steps}')
print(f'FLOPs/step: {flops_per_step:,}')

## Profile Step 1: Kernel Definition [CPU]

RawKernel constructor — lazy, no compilation yet.

In [ ]:
cp.cuda.Device().synchronize()
t0_step = time.perf_counter()

cupy_update_H = cp.RawKernel(r'''
extern "C" __global__
void update_H(float* Ez, float* Hx, float* Hy,
              float coeff_hx, float coeff_hy, int Nx, int Ny) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    int j = blockIdx.y * blockDim.y + threadIdx.y;
    if (i < Nx && j < Ny - 1) { int idx=i*Ny+j; Hx[idx] -= coeff_hx*(Ez[idx+1]-Ez[idx]); }
    if (i < Nx-1 && j < Ny)   { int idx=i*Ny+j; Hy[idx] += coeff_hy*(Ez[idx+Ny]-Ez[idx]); }
}
''', 'update_H')

cupy_update_Ez = cp.RawKernel(r'''
extern "C" __global__
void update_Ez(float* Ez, const float* Hx, const float* Hy,
               float coeff_ez, float dx, float dy,
               int Nx, int Ny, int src_x, int src_y,
               const float* pulse_arr, int step) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    int j = blockIdx.y * blockDim.y + threadIdx.y;
    if (i >= Nx || j >= Ny) return;
    int idx = i*Ny+j;
    if (i >= 1 && j >= 1) {
        float dHy = (Hy[idx]-Hy[idx-Ny])/dx;
        float dHx = (Hx[idx]-Hx[idx-1])/dy;
        Ez[idx] += coeff_ez*(dHy-dHx);
    }
    if (i==src_x && j==src_y) Ez[idx] = pulse_arr[step];
    if (i==0||i==Nx-1||j==0||j==Ny-1) Ez[idx] = 0.0f;
}
''', 'update_Ez')

cupy_energy = cp.RawKernel(r'''
extern "C" __global__
void compute_energy(const float* Ez, const float* Hx, const float* Hy,
                    float* out, float eps0, float mu0,
                    float dx, float dy, int Nx, int Ny) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    int j = blockIdx.y * blockDim.y + threadIdx.y;
    if (i >= Nx || j >= Ny) return;
    int idx = i*Ny+j;
    float ez=Ez[idx], hx=Hx[idx], hy=Hy[idx];
    float e = (0.5f*eps0*ez*ez + 0.5f*mu0*(hx*hx+hy*hy))*dx*dy;
    atomicAdd(out, e);
}
''', 'compute_energy')

t_kernel_def = time.perf_counter() - t0_step
profile['setup']['kernel_definition_ms'] = t_kernel_def * 1000
print(f'Step 1 — Kernel definition: {t_kernel_def*1000:.3f} ms [CPU, lazy — no compilation yet]')

## Profile Step 2: GPU Memory Allocation [GPU alloc]

In [ ]:
cp.cuda.Device().synchronize()
t0_step = time.perf_counter()

Ez_cu = cp.zeros((Nx, Ny), dtype=cp.float32)
Hx_cu = cp.zeros((Nx, Ny), dtype=cp.float32)
Hy_cu = cp.zeros((Nx, Ny), dtype=cp.float32)
energy_cu = cp.zeros(1, dtype=cp.float32)

cp.cuda.Device().synchronize()
t_alloc = time.perf_counter() - t0_step
profile['setup']['gpu_alloc_ms'] = t_alloc * 1000
gpu_bytes = 3 * Nx * Ny * 4 + 4
print(f'Step 2 — GPU allocation: {t_alloc*1000:.3f} ms [{gpu_bytes/1024:.1f} KB on GPU]')

## Profile Step 3: Pulse Compute (CPU) + H→D Upload

In [ ]:
# --- CPU compute ---
cp.cuda.Device().synchronize()
t0_step = time.perf_counter()

pulse_np = np.array([np.exp(-((n*dt - t0)/spread)**2)
                     for n in range(n_steps)], dtype=np.float32)

t_pulse_cpu = time.perf_counter() - t0_step
profile['setup']['pulse_cpu_compute_ms'] = t_pulse_cpu * 1000

# --- H→D upload ---
t0_step = time.perf_counter()
pulse_cu = cp.array(pulse_np, dtype=cp.float32)
cp.cuda.Device().synchronize()
t_pulse_upload = time.perf_counter() - t0_step
profile['setup']['pulse_h2d_upload_ms'] = t_pulse_upload * 1000

print(f'Step 3a — Pulse CPU compute: {t_pulse_cpu*1000:.3f} ms [400 × exp() on CPU]')
print(f'Step 3b — Pulse H→D upload:  {t_pulse_upload*1000:.3f} ms [{len(pulse_np)*4} bytes CPU→GPU]')

## Profile Step 4: Scalar Casting + Launch Config [CPU]

In [ ]:
t0_step = time.perf_counter()

_chx  = np.float32(dt / (mu0 * dy))
_chy  = np.float32(dt / (mu0 * dx))
_cez  = np.float32(dt / eps0)
_dx   = np.float32(dx)
_dy   = np.float32(dy)
_eps0 = np.float32(eps0)
_mu0  = np.float32(mu0)
_Nx   = np.int32(Nx)
_Ny   = np.int32(Ny)
_sx   = np.int32(src_x)
_sy   = np.int32(src_y)

BLOCK = (8, 32)
GRID  = ((Nx + 8 - 1) // 8, (Ny + 32 - 1) // 32)

t_scalars = time.perf_counter() - t0_step
profile['setup']['scalar_cast_ms'] = t_scalars * 1000
print(f'Step 4 — Scalar cast + config: {t_scalars*1000:.3f} ms [CPU, 11 scalars + grid/block]')

## Profile Step 5: First Kernel Launch (JIT Compilation)

The first `__call__()` on each RawKernel triggers CUDA compilation.  
This is the most expensive one-time cost.

In [ ]:
# --- First launch of update_H (triggers compilation) ---
cp.cuda.Device().synchronize()
t0_step = time.perf_counter()

cupy_update_H(GRID, BLOCK, (Ez_cu, Hx_cu, Hy_cu, _chx, _chy, _Nx, _Ny))
cp.cuda.Device().synchronize()

t_first_H = time.perf_counter() - t0_step
profile['setup']['first_launch_H_ms'] = t_first_H * 1000

# --- First launch of update_Ez (triggers compilation) ---
cp.cuda.Device().synchronize()
t0_step = time.perf_counter()

cupy_update_Ez(GRID, BLOCK, (Ez_cu, Hx_cu, Hy_cu, _cez, _dx, _dy,
                              _Nx, _Ny, _sx, _sy, pulse_cu, np.int32(0)))
cp.cuda.Device().synchronize()

t_first_Ez = time.perf_counter() - t0_step
profile['setup']['first_launch_Ez_ms'] = t_first_Ez * 1000

# --- First launch of energy (triggers compilation) ---
cp.cuda.Device().synchronize()
t0_step = time.perf_counter()

energy_cu[:] = 0
cupy_energy(GRID, BLOCK, (Ez_cu, Hx_cu, Hy_cu, energy_cu,
                           _eps0, _mu0, _dx, _dy, _Nx, _Ny))
cp.cuda.Device().synchronize()

t_first_energy = time.perf_counter() - t0_step
profile['setup']['first_launch_energy_ms'] = t_first_energy * 1000

print(f'Step 5 — First launch (JIT compilation):')
print(f'  update_H:       {t_first_H*1000:.1f} ms [compile + execute]')
print(f'  update_Ez:      {t_first_Ez*1000:.1f} ms [compile + execute]')
print(f'  compute_energy: {t_first_energy*1000:.1f} ms [compile + execute]')
print(f'  Total JIT:      {(t_first_H+t_first_Ez+t_first_energy)*1000:.1f} ms')

# Reset fields after profiling first launches
Ez_cu[:] = 0; Hx_cu[:] = 0; Hy_cu[:] = 0; energy_cu[:] = 0
cp.cuda.Device().synchronize()

## Profile Step 6: Simulation Loop — Per-Step Breakdown

For **every step**, we separately time:
- `cupy_update_H` launch + GPU execution
- `cupy_update_Ez` launch + GPU execution
- `energy_cu[:] = 0` memset
- `cupy_energy` launch + GPU execution
- `cp.cuda.Device().synchronize()` sync stall
- `float(energy_cu.get()[0])` D→H 4 bytes
- (on snapshot steps) `cp.asnumpy(Ez_cu).copy()` D→H 160 KB

In [ ]:
total_energy_prof = np.zeros(n_steps)
snapshots_prof = {}

print(f'Profiling {n_steps} steps with per-operation timing ...')
print(f'(This is slower than normal due to sync after every operation)\n')

t_loop_start = time.perf_counter()

for n in range(n_steps):
    step_profile = {'step': n}

    # ── H kernel ──
    cp.cuda.Device().synchronize()
    t0_op = time.perf_counter()
    cupy_update_H(GRID, BLOCK, (Ez_cu, Hx_cu, Hy_cu, _chx, _chy, _Nx, _Ny))
    cp.cuda.Device().synchronize()
    step_profile['H_kernel_ms'] = (time.perf_counter() - t0_op) * 1000

    # ── Ez kernel ──
    cp.cuda.Device().synchronize()
    t0_op = time.perf_counter()
    cupy_update_Ez(GRID, BLOCK, (Ez_cu, Hx_cu, Hy_cu, _cez, _dx, _dy,
                                  _Nx, _Ny, _sx, _sy, pulse_cu, np.int32(n)))
    cp.cuda.Device().synchronize()
    step_profile['Ez_kernel_ms'] = (time.perf_counter() - t0_op) * 1000

    # ── Energy memset ──
    cp.cuda.Device().synchronize()
    t0_op = time.perf_counter()
    energy_cu[:] = 0
    cp.cuda.Device().synchronize()
    step_profile['energy_memset_ms'] = (time.perf_counter() - t0_op) * 1000

    # ── Energy kernel ──
    cp.cuda.Device().synchronize()
    t0_op = time.perf_counter()
    cupy_energy(GRID, BLOCK, (Ez_cu, Hx_cu, Hy_cu, energy_cu,
                               _eps0, _mu0, _dx, _dy, _Nx, _Ny))
    cp.cuda.Device().synchronize()
    step_profile['energy_kernel_ms'] = (time.perf_counter() - t0_op) * 1000

    # ── D→H energy readback ──
    t0_op = time.perf_counter()
    total_energy_prof[n] = float(energy_cu.get()[0])
    step_profile['energy_d2h_ms'] = (time.perf_counter() - t0_op) * 1000

    # ── D→H snapshot (only on snapshot steps) ──
    if n in snapshot_steps:
        t0_op = time.perf_counter()
        snapshots_prof[n] = cp.asnumpy(Ez_cu).copy()
        step_profile['snapshot_d2h_ms'] = (time.perf_counter() - t0_op) * 1000
        step_profile['is_snapshot'] = True
    else:
        step_profile['is_snapshot'] = False

    # Total step time
    step_profile['total_step_ms'] = (
        step_profile['H_kernel_ms'] + step_profile['Ez_kernel_ms'] +
        step_profile['energy_memset_ms'] + step_profile['energy_kernel_ms'] +
        step_profile['energy_d2h_ms'] +
        step_profile.get('snapshot_d2h_ms', 0)
    )

    profile['per_step'].append(step_profile)

    if n % 100 == 0 or n == n_steps - 1:
        print(f'  step {n:4d}/{n_steps}  |  '
              f'H={step_profile["H_kernel_ms"]:.3f}  '
              f'Ez={step_profile["Ez_kernel_ms"]:.3f}  '
              f'E={step_profile["energy_kernel_ms"]:.3f}  '
              f'd2h={step_profile["energy_d2h_ms"]:.3f} ms'
              f'{"  +snap" if step_profile["is_snapshot"] else ""}')

t_loop_total = time.perf_counter() - t_loop_start
profile['setup']['total_loop_ms'] = t_loop_total * 1000
print(f'\nTotal profiled loop: {t_loop_total*1000:.1f} ms')
print(f'(Includes sync overhead from profiling — real run is faster)')

## Aggregate Statistics

In [ ]:
import statistics

# Skip step 0 (may include leftover JIT cache effects)
steps = profile['per_step'][1:]

ops = ['H_kernel_ms', 'Ez_kernel_ms', 'energy_memset_ms',
       'energy_kernel_ms', 'energy_d2h_ms', 'total_step_ms']

agg = {}
print(f'{"Operation":<20} {"Mean":>8} {"Median":>8} {"Min":>8} {"Max":>8} {"Stdev":>8}  (ms)')
print(f'{"-"*20} {"-"*8} {"-"*8} {"-"*8} {"-"*8} {"-"*8}')

for op in ops:
    vals = [s[op] for s in steps]
    stats = {
        'mean': statistics.mean(vals),
        'median': statistics.median(vals),
        'min': min(vals),
        'max': max(vals),
        'stdev': statistics.stdev(vals) if len(vals) > 1 else 0,
    }
    agg[op] = stats
    print(f'{op:<20} {stats["mean"]:>8.4f} {stats["median"]:>8.4f} '
          f'{stats["min"]:>8.4f} {stats["max"]:>8.4f} {stats["stdev"]:>8.4f}')

# Snapshot stats
snap_vals = [s['snapshot_d2h_ms'] for s in steps if s.get('is_snapshot')]
if snap_vals:
    stats = {
        'mean': statistics.mean(snap_vals),
        'median': statistics.median(snap_vals),
        'min': min(snap_vals),
        'max': max(snap_vals),
        'stdev': statistics.stdev(snap_vals) if len(snap_vals) > 1 else 0,
    }
    agg['snapshot_d2h_ms'] = stats
    print(f'{"snapshot_d2h_ms":<20} {stats["mean"]:>8.4f} {stats["median"]:>8.4f} '
          f'{stats["min"]:>8.4f} {stats["max"]:>8.4f} {stats["stdev"]:>8.4f}')

profile['aggregate'] = agg

# Time budget
total_mean = agg['total_step_ms']['mean']
print(f'\nTime budget per step ({total_mean:.4f} ms):')
for op in ops[:-1]:
    pct = agg[op]['mean'] / total_mean * 100
    bar = '#' * int(pct / 2)
    print(f'  {op:<20} {agg[op]["mean"]:>7.4f} ms  ({pct:>5.1f}%)  {bar}')

## Save Profile to JSON

In [ ]:
# Add setup summary
profile['setup']['total_setup_ms'] = (
    profile['setup']['kernel_definition_ms'] +
    profile['setup']['gpu_alloc_ms'] +
    profile['setup']['pulse_cpu_compute_ms'] +
    profile['setup']['pulse_h2d_upload_ms'] +
    profile['setup']['scalar_cast_ms'] +
    profile['setup']['first_launch_H_ms'] +
    profile['setup']['first_launch_Ez_ms'] +
    profile['setup']['first_launch_energy_ms']
)

out_path = 'cupy_profile_results.json'
with open(out_path, 'w') as f:
    json.dump(profile, f, indent=2)

print(f'Profile saved to {out_path}')
print(f'  Setup entries:  {len(profile["setup"])}')
print(f'  Per-step entries: {len(profile["per_step"])}')
print(f'  Aggregate ops:  {len(profile["aggregate"])}')
print(f'  File size: {len(json.dumps(profile))/1024:.1f} KB')

## Visualization: Per-Operation Time Across All Steps

In [ ]:
import matplotlib.pyplot as plt

steps_range = np.arange(n_steps)
H_times  = [s['H_kernel_ms']  for s in profile['per_step']]
Ez_times = [s['Ez_kernel_ms'] for s in profile['per_step']]
En_times = [s['energy_kernel_ms'] for s in profile['per_step']]
d2h_times = [s['energy_d2h_ms'] for s in profile['per_step']]
memset_times = [s['energy_memset_ms'] for s in profile['per_step']]

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# ── Panel 1: Kernel times ──
axes[0].plot(steps_range, H_times, 'b-', lw=0.7, alpha=0.7, label='H kernel')
axes[0].plot(steps_range, Ez_times, 'r-', lw=0.7, alpha=0.7, label='Ez kernel')
axes[0].plot(steps_range, En_times, 'g-', lw=0.7, alpha=0.7, label='Energy kernel')
axes[0].set_ylabel('Time [ms]')
axes[0].set_title('GPU Kernel Execution Time per Step')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# ── Panel 2: Sync + D→H ──
axes[1].plot(steps_range, d2h_times, 'm-', lw=0.7, alpha=0.7, label='D→H energy (4B)')
axes[1].plot(steps_range, memset_times, 'c-', lw=0.7, alpha=0.7, label='Memset')
# Mark snapshots
snap_x = [s['step'] for s in profile['per_step'] if s.get('is_snapshot')]
snap_y = [s.get('snapshot_d2h_ms', 0) for s in profile['per_step'] if s.get('is_snapshot')]
axes[1].scatter(snap_x, snap_y, c='red', s=40, zorder=5, label=f'D→H snapshot (160KB, {len(snap_x)}x)')
axes[1].set_ylabel('Time [ms]')
axes[1].set_title('Sync & Transfer Time per Step')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

# ── Panel 3: Total per step ──
total_times = [s['total_step_ms'] for s in profile['per_step']]
axes[2].plot(steps_range, total_times, 'k-', lw=0.7, alpha=0.7)
axes[2].axhline(y=agg['total_step_ms']['mean'], color='red', ls='--', lw=1, label=f'Mean: {agg["total_step_ms"]["mean"]:.3f} ms')
axes[2].set_xlabel('Step')
axes[2].set_ylabel('Time [ms]')
axes[2].set_title('Total Time per Step')
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3)

fig.suptitle(f'CuPy FDTD Per-Step Profile — {gpu_name} ({Nx}x{Ny}, {n_steps} steps)',
             fontsize=13, fontweight='bold')
fig.tight_layout()
plt.savefig('cupy_profile_per_step.png', dpi=150)
plt.show()
print('Saved: cupy_profile_per_step.png')

## Visualization: Time Budget Breakdown

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: Per-step pie chart ──
labels = ['H kernel', 'Ez kernel', 'Energy memset', 'Energy kernel', 'D→H energy']
sizes = [agg[op]['mean'] for op in ops[:-1]]
colors = ['#3498db', '#e74c3c', '#95a5a6', '#2ecc71', '#9b59b6']
explode = [0, 0, 0, 0, 0.05]

wedges, texts, autotexts = ax1.pie(sizes, labels=labels, autopct='%1.1f%%',
                                    colors=colors, explode=explode, startangle=90,
                                    textprops={'fontsize': 9})
ax1.set_title(f'Per-Step Time Budget\n(mean = {total_mean:.3f} ms/step)', fontweight='bold')

# ── Right: Setup breakdown bar chart ──
setup_items = [
    ('Kernel def', profile['setup']['kernel_definition_ms']),
    ('GPU alloc', profile['setup']['gpu_alloc_ms']),
    ('Pulse CPU', profile['setup']['pulse_cpu_compute_ms']),
    ('Pulse H→D', profile['setup']['pulse_h2d_upload_ms']),
    ('Scalars', profile['setup']['scalar_cast_ms']),
    ('JIT: H', profile['setup']['first_launch_H_ms']),
    ('JIT: Ez', profile['setup']['first_launch_Ez_ms']),
    ('JIT: Energy', profile['setup']['first_launch_energy_ms']),
]

names = [x[0] for x in setup_items]
vals = [x[1] for x in setup_items]
bar_colors = ['#bdc3c7'] * 5 + ['#e67e22'] * 3  # gray for fast, orange for JIT

bars = ax2.barh(names, vals, color=bar_colors, edgecolor='black', lw=0.5)
for bar, v in zip(bars, vals):
    label = f'{v:.1f} ms' if v >= 1 else f'{v*1000:.0f} μs'
    ax2.text(bar.get_width() + max(vals)*0.02, bar.get_y() + bar.get_height()/2,
             label, va='center', fontsize=8)

ax2.set_xlabel('Time [ms]')
ax2.set_title(f'One-Time Setup Costs\n(total = {profile["setup"]["total_setup_ms"]:.1f} ms)', fontweight='bold')
ax2.invert_yaxis()

fig.suptitle(f'CuPy FDTD Profile — {gpu_name}', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.savefig('cupy_profile_breakdown.png', dpi=150)
plt.show()
print('Saved: cupy_profile_breakdown.png')

## Visualization: Step 0 vs Steady-State

In [ ]:
# Compare step 0 (may have JIT residual) vs median step
s0 = profile['per_step'][0]

fig, ax = plt.subplots(figsize=(10, 4))

op_names = ['H kernel', 'Ez kernel', 'Energy memset', 'Energy kernel', 'D→H energy']
op_keys  = ['H_kernel_ms', 'Ez_kernel_ms', 'energy_memset_ms', 'energy_kernel_ms', 'energy_d2h_ms']

x = np.arange(len(op_names))
w = 0.35

vals_step0 = [s0[k] for k in op_keys]
vals_median = [agg[k]['median'] for k in op_keys]

bars1 = ax.bar(x - w/2, vals_step0, w, label=f'Step 0', color='#e74c3c', edgecolor='black', lw=0.5)
bars2 = ax.bar(x + w/2, vals_median, w, label=f'Median (steps 1-{n_steps-1})', color='#2ecc71', edgecolor='black', lw=0.5)

for bar, v in zip(bars1, vals_step0):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{v:.3f}', ha='center', va='bottom', fontsize=7)
for bar, v in zip(bars2, vals_median):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{v:.3f}', ha='center', va='bottom', fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels(op_names, fontsize=9)
ax.set_ylabel('Time [ms]')
ax.set_title(f'Step 0 vs Steady-State — {gpu_name}')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

fig.tight_layout()
plt.savefig('cupy_profile_step0_vs_steady.png', dpi=150)
plt.show()
print('Saved: cupy_profile_step0_vs_steady.png')

## Summary

In [ ]:
print(f"{'='*65}")
print(f'  CuPy FDTD Profile Summary — {gpu_name}')
print(f'  {Nx}x{Ny} grid, {n_steps} steps')
print(f"{'='*65}")
print(f'')
print(f'  ONE-TIME SETUP COSTS:')
print(f'    Kernel definition (lazy):    {profile["setup"]["kernel_definition_ms"]:.3f} ms')
print(f'    GPU memory allocation:       {profile["setup"]["gpu_alloc_ms"]:.3f} ms')
print(f'    Pulse compute (CPU):         {profile["setup"]["pulse_cpu_compute_ms"]:.3f} ms')
print(f'    Pulse H→D upload:            {profile["setup"]["pulse_h2d_upload_ms"]:.3f} ms')
print(f'    Scalar cast:                 {profile["setup"]["scalar_cast_ms"]:.3f} ms')
print(f'    JIT compile (first launch):  {(profile["setup"]["first_launch_H_ms"]+profile["setup"]["first_launch_Ez_ms"]+profile["setup"]["first_launch_energy_ms"]):.1f} ms')
print(f'    Total setup:                 {profile["setup"]["total_setup_ms"]:.1f} ms')
print(f'')
print(f'  PER-STEP COSTS (median, ms):')
for op, name in zip(ops[:-1], ['H kernel', 'Ez kernel', 'Energy memset', 'Energy kernel', 'D→H energy']):
    print(f'    {name:<24} {agg[op]["median"]:.4f} ms')
print(f'    {"Total/step":<24} {agg["total_step_ms"]["median"]:.4f} ms')
print(f'')
print(f'  BOTTLENECK ANALYSIS:')
total = agg['total_step_ms']['mean']
compute = agg['H_kernel_ms']['mean'] + agg['Ez_kernel_ms']['mean']
overhead = agg['energy_memset_ms']['mean'] + agg['energy_kernel_ms']['mean'] + agg['energy_d2h_ms']['mean']
print(f'    GPU compute (H+Ez):   {compute:.4f} ms  ({compute/total*100:.1f}%)')
print(f'    Energy overhead:      {overhead:.4f} ms  ({overhead/total*100:.1f}%)')
print(f'')
print(f'  Saved files:')
print(f'    cupy_profile_results.json       — raw data ({len(json.dumps(profile))/1024:.1f} KB)')
print(f'    cupy_profile_per_step.png        — per-step time series')
print(f'    cupy_profile_breakdown.png       — pie chart + setup bars')
print(f'    cupy_profile_step0_vs_steady.png — JIT warmup effect')
print(f"{'='*65}")